# Workshop Preparation: Dataset + Neuron Graph Pre-Compilation

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>Workshop Organizers Only:</strong> Run this notebook <strong>once</strong> before the workshop.
Participants do NOT need to run this — the prepared artifacts will already be in S3.
</div>

## What This Does

1. **Tokenizes the training dataset** and uploads to S3
2. **Pre-compiles Neuron graphs** on a high-memory instance and caches to S3

Both steps produce S3 artifacts that `01_fine-tuning.ipynb` consumes directly,
giving participants a fast experience (~5 min training, no compilation wait).

## Why Pre-Compile?

The Neuron compiler (`neuronx-cc`) compiles PyTorch computation graphs into hardware-specific
binaries (`.neff` files) for Trainium chips. This compilation requires more host memory
than the 32GB available on `ml.trn1.2xlarge` and takes 10-30+ minutes.

By pre-compiling on `ml.trn1.2xlarge` (512GB RAM, 32 NeuronCores) and caching to S3, the training job
loads pre-compiled graphs and starts training immediately.

**Estimated time:** ~20-30 minutes total (one-time cost)

---
## 1. Setup

In [ ]:
# --- Lab dependencies (managed via uv) ---------------------------------------
# Installs THIS lab's complete, self-contained kernel dependencies from the
# lab requirements.txt using uv. Idempotent and fast when already satisfied.
# This is the only dependency step the lab needs - Setup.ipynb is not required.
import sys
!pip install -q uv
!uv pip install -q --python {sys.executable} -r requirements.txt

In [ ]:
import os
import json
import boto3
import sagemaker
from config import MODEL_ID
from sagemaker.core.helper.session_helper import Session, get_execution_role
from importlib.metadata import version

boto_session = boto3.Session()
sagemaker_session = Session(boto_session)
role = get_execution_role()

bucket = sagemaker_session.default_bucket()
region = boto_session.region_name

# S3 paths
S3_PREFIX = "lab8-trainium-inferentia"
NEURON_CACHE_S3_URI = f"s3://{bucket}/{S3_PREFIX}/neuron-cache"

print(f"SageMaker SDK version: {version('sagemaker')}")
print(f"Role: {role}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")
print(f"Model: {MODEL_ID}")

In [ ]:
# Training configuration — must match 01_fine-tuning.ipynb
DATASET_ID = "databricks/databricks-dolly-15k"
MAX_SEQ_LENGTH = 512
NUM_TRAIN_SAMPLES = 1000
BATCH_SIZE = 2
LORA_R = 16
LORA_ALPHA = 16
GRADIENT_ACCUMULATION_STEPS = 4
NUM_WORKERS = 1  # nproc_per_node=1 (single process gets full 32GB host RAM for compiler)

print(f"Dataset: {DATASET_ID}")
print(f"Sequence length: {MAX_SEQ_LENGTH}")
print(f"Training samples: {NUM_TRAIN_SAMPLES}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Num workers : {NUM_WORKERS}")

---
## 2. Prepare and Upload Dataset

We use the [Databricks Dolly 15k](https://huggingface.co/datasets/databricks/databricks-dolly-15k) dataset.
Steps:
1. Format each sample into the model's chat template
2. Tokenize with padding/truncation to `MAX_SEQ_LENGTH`
3. Take a subset and split into train/eval
4. Upload to S3

In [ ]:
from datasets import load_dataset

# Load the raw Dolly dataset (NeuronSFTTrainer handles tokenization internally)
dataset = load_dataset(DATASET_ID, split="train")

# Take a subset for workshop speed
dataset = dataset.shuffle(seed=42).select(range(NUM_TRAIN_SAMPLES))

print(f"Dataset size: {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")
print(f"Sample: {dataset[0]["instruction"][:100]}...")

In [ ]:
# Save raw dataset and upload to S3
# NeuronSFTTrainer will format and tokenize during training using formatting_func
local_train_path = "datasets/train"

dataset.save_to_disk(local_train_path)
train_s3_uri = sagemaker_session.upload_data(local_train_path, bucket=bucket, key_prefix=f"{S3_PREFIX}/datasets/train")

print(f"Training data uploaded to: {train_s3_uri}")
print(f"Dataset preparation complete!")

---
## 3. Pre-Compile Neuron Graphs

We use a SageMaker Training job on `ml.trn1.2xlarge` (512GB RAM, 32 NeuronCores) with the same
Neuron DLC used for training. The job runs `neuron_parallel_compile` which:

1. Traces the training script with dummy data (discovers graph shapes)
2. Compiles all graph segments using the Neuron compiler
3. Saves compiled `.neff` files to S3

This takes ~15-20 minutes but only needs to be done once.

In [ ]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, SourceCode
from sagemaker.train.distributed import Torchrun
from sagemaker.core.shapes import OutputDataConfig

# Same DLC as the training job - ensures matching neuronx-cc version
TRAINING_IMAGE = f"763104351884.dkr.ecr.{region}.amazonaws.com/huggingface-pytorch-training-neuronx:2.8.0-transformers4.55.4-neuronx-py310-sdk2.26.0-ubuntu22.04"
PROCESSING_INSTANCE_TYPE = "ml.trn1.2xlarge"  # 8 vCPU, 32GB RAM, 2 NeuronCores

print(f"Processing image: {TRAINING_IMAGE.split(chr(47))[-1]}")
print(f"Instance: {PROCESSING_INSTANCE_TYPE}")
print(f"Model to compile: {MODEL_ID}")

In [ ]:
# Launch pre-compilation using ModelTrainer on trn1.2xlarge.
# Uses the REAL training data to ensure all graph shapes are captured.
# neuron_parallel_compile traces the training script and compiles all graph modules.

from sagemaker.train.configs import InputData

model_short = MODEL_ID.split("/")[-1]
cache_s3_uri = f"s3://{bucket}/{S3_PREFIX}/neuron-cache/{model_short}"

print(f"Pre-compiling: {MODEL_ID}")
print(f"Cache destination: {cache_s3_uri}")
print(f"Training data: {train_s3_uri}")

compiler = ModelTrainer(
    training_image=TRAINING_IMAGE,
    sagemaker_session=sagemaker_session,
    source_code=SourceCode(
        source_dir="src",
        entry_script="precompile.py",
    ),
    compute=Compute(
        instance_type=PROCESSING_INSTANCE_TYPE,
        instance_count=1,
        volume_size_in_gb=100,
    ),
    role=role,
    base_job_name="neuron-precompile",
    output_data_config=OutputDataConfig(s3_output_path=cache_s3_uri),
    hyperparameters={
        "model_id": MODEL_ID,
        "max_seq_length": MAX_SEQ_LENGTH,
        "batch_size": BATCH_SIZE,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "num_workers": NUM_WORKERS,
        "cache_dir": "/opt/ml/model",
    },
    environment={
        "MALLOC_ARENA_MAX": "64",
        "NEURON_FUSE_SOFTMAX": "1",
        "NEURON_CC_FLAGS": "--model-type=transformer --distribution-strategy=llm-training",
    },
)

# Pass real training data so graph shapes match actual training
compiler.train(
    input_data_config=[
        InputData(channel_name="train", data_source=train_s3_uri),
    ]
)

print(f"\nPre-compilation complete!")
print(f"Cache uploaded to: {cache_s3_uri}")

---
## 4. Verify Artifacts

In [ ]:
s3 = boto3.client('s3', region_name=region)

# Check dataset
print("=== Dataset (S3) ===")
for split_name in ['train']:
    prefix = f"{S3_PREFIX}/datasets/{split_name}"
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    count = response.get('KeyCount', 0)
    print(f"  {split_name}: {count} files")

# Check Neuron cache
print(f"\n=== Neuron Compile Cache (S3) ===")
prefix = f"{S3_PREFIX}/neuron-cache"
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
neff_files = [obj for obj in response.get('Contents', []) if obj['Key'].endswith('.neff')]
total_size = sum(obj['Size'] for obj in neff_files)

print(f"  .neff files: {len(neff_files)}")
print(f"  Total cache size: {total_size / (1024 * 1024):.1f} MB")

print(f"\n=== S3 Paths for 01_fine-tuning.ipynb ===")
print(f"  Train data: s3://{bucket}/{S3_PREFIX}/datasets/train")
print(f"  Neuron cache: {NEURON_CACHE_S3_URI}")

---
## 5. Summary

Workshop preparation is complete. The following S3 artifacts are ready:

| Artifact | S3 Path | Purpose |
|----------|---------|--------|
| Training data | `s3://<bucket>/lab8-trainium-inferentia/datasets/train` | Tokenized train split |
| Neuron cache | `s3://<bucket>/lab8-trainium-inferentia/neuron-cache` | Pre-compiled .neff files |

Participants running `01_fine-tuning.ipynb` will:
1. Reference the pre-uploaded dataset from S3 (no tokenization needed)
2. Load pre-compiled Neuron graphs via `NEURON_COMPILE_CACHE_URL`
3. Start training immediately (~5 min on `ml.trn1.2xlarge`)

**Re-run this notebook if you change:**
- Model ID
- Sequence length or batch size
- Number of NeuronCores (nproc_per_node)
- Neuron SDK version (DLC image tag)